In [1]:
import pandas as pd
import numpy as np
import torch
import random
from itables import show
import pubchempy as pcp
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from descriptastorus.descriptors import rdDescriptors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from functools import lru_cache
import xgboost
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, max_error, mean_absolute_percentage_error, r2_score

In [2]:
def get_regression_metrics(model, X, y_true):
    """
    Get a dicionary with regression metrics,
    including MAE, MSE, Max Error, MAPE, R2 Score
    """
    y_predicted = model.predict(X)
    mae = mean_absolute_error(y_true, y_predicted)
    mse = mean_squared_error(y_true, y_predicted)
    maximum_error = max_error(y_true, y_predicted)
    mape = mean_absolute_percentage_error(y_true, y_predicted)
    r2 = r2_score(y_true, y_predicted)
    metrics_dict = {
        'mae': mae,
        'mse': mse,
        'max_error': maximum_error,
        'mape': mape,
        'r2': r2
    }
    return metrics_dict

In [3]:
def dummy_regressor_baseline(X_train, y_train):
    """
    Build Dummy Regressor Baseline Models.
    """
    dummyregressor_mean = DummyRegressor(strategy='mean')
    dummyregressor_median = DummyRegressor(strategy='median')
    dummyregressor_mean.fit(X_train, y_train)
    dummyregressor_median.fit(X_train, y_train)
    return dummyregressor_mean, dummyregressor_median

In [5]:
train_df = pd.read_csv('train_df.csv')
test_df = pd.read_csv('test_df.csv')
val_df = pd.read_csv('val_df.csv')

In [8]:
TARGET_VARIABLE = 'Mole fraction'

In [9]:
X_train = train_df.drop(columns=[TARGET_VARIABLE])
y_train = train_df[TARGET_VARIABLE] 
X_test = test_df.drop(columns=[TARGET_VARIABLE])
y_test = test_df[TARGET_VARIABLE]
X_val = val_df.drop(columns=[TARGET_VARIABLE])
y_val = val_df[TARGET_VARIABLE]

In [11]:
X_train.drop(columns=['Component 1', 'Component 2',"Smiles 1","Smiles 2", 'mol1', 'mol2', 'strat_key','combination_counts'], inplace=True)
X_test.drop(columns=['Component 1', 'Component 2',"Smiles 1","Smiles 2", 'mol1', 'mol2', 'strat_key','combination_counts'], inplace=True)
X_val.drop(columns=['Component 1', 'Component 2',"Smiles 1","Smiles 2", 'mol1', 'mol2', 'strat_key','combination_counts'], inplace=True)

In [13]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
X_val.to_csv('X_val.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
y_val.to_csv('y_val.csv', index=False)

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(101)
np.random.seed(101)
random.seed(101)

In [12]:
X_train.head()

,property,value,phase,"Temperature, K","Pressure, kPa","('BalabanJ', <class 'numpy.float64'>)","('BertzCT', <class 'numpy.float64'>)","('Chi0', <class 'numpy.float64'>)","('Chi0n', <class 'numpy.float64'>)","('Chi0v', <class 'numpy.float64'>)",...,"('fr_sulfonamd', <class 'numpy.float64'>).1","('fr_sulfone', <class 'numpy.float64'>).1","('fr_term_acetylene', <class 'numpy.float64'>).1","('fr_tetrazole', <class 'numpy.float64'>).1","('fr_thiazole', <class 'numpy.float64'>).1","('fr_thiocyan', <class 'numpy.float64'>).1","('fr_thiophene', <class 'numpy.float64'>).1","('fr_unbrch_alkane', <class 'numpy.float64'>).1","('fr_urea', <class 'numpy.float64'>).1","('qed', <class 'numpy.float64'>).1"
0,"Mass density, kg/m3",695.300000,Liquid,362.710,3000.0,0.948491,0.001101,0.003896,1.882147e-02,1.231789e-02,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.238586
1,"Mass density, kg/m3",855.400000,Liquid,298.150,35000.0,0.992385,0.004673,0.009433,1.973585e-02,1.294688e-02,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.308214
2,"Isobaric coefficient of expansion, 1/K",0.000791,Gas,278.150,45000.0,0.918955,0.000793,0.000581,1.603592e-03,1.017670e-03,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,9.607067e-01,0.166633,0.307141
3,"Viscosity, Pa*s",0.000018,Gas,323.216,398.6,0.984555,0.000816,0.000009,2.329201e-08,4.812511e-07,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.179187
4,"Mass density, kg/m3",651.800000,Liquid,353.150,10000.0,0.404909,0.000577,0.000009,4.958730e-06,7.975824e-06,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,9.991992e-01,0.166633,0.324618


In [15]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

(85106, 405)
(85106,)
(10639, 405)
(10639,)
(10638, 405)
(10638,)


In [ ]:
xgboost_param_grid = {
                    'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2],
                    'n_estimators': [100, 200, 400, 600, 800, 1000],
                    'max_depth': [3, 4, 5, 6],
                    'min_child_weight': [1, 3, 5],
                    'subsample': [0.6,0.8,1.0],
                    'colsample_bytree': [0.6, 0.8, 1.0]
            }
